# 第 7 讲：综合评价模型

> 落成评价模型示例代码：指标标准化、熵权法、TOPSIS 和排名解释。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-07"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

In [ ]:
alternatives = [f"方案{i}" for i in range(1, 7)]
X = pd.DataFrame({
    "成本": [72, 65, 80, 68, 75, 70],
    "效率": [88, 82, 91, 85, 89, 83],
    "可靠性": [0.92, 0.88, 0.95, 0.90, 0.93, 0.86],
    "建设周期": [14, 10, 18, 12, 16, 11],
}, index=alternatives)
directions = {"成本": "min", "效率": "max", "可靠性": "max", "建设周期": "min"}

def minmax_normalize(df, directions):
    out = pd.DataFrame(index=df.index)
    for col in df.columns:
        cmin, cmax = df[col].min(), df[col].max()
        if directions[col] == "max":
            out[col] = (df[col] - cmin) / (cmax - cmin)
        else:
            out[col] = (cmax - df[col]) / (cmax - cmin)
    return out

norm = minmax_normalize(X, directions)
P = norm / norm.sum(axis=0)
eps = 1e-12
entropy = -(P * np.log(P + eps)).sum(axis=0) / np.log(len(norm))
weights = (1 - entropy) / (1 - entropy).sum()
weights.to_csv(OUTPUT_ROOT / "entropy_weights.csv", encoding="utf-8-sig")
weights

In [ ]:
weighted = norm * weights
positive = weighted.max(axis=0)
negative = weighted.min(axis=0)
d_pos = np.sqrt(((weighted - positive) ** 2).sum(axis=1))
d_neg = np.sqrt(((weighted - negative) ** 2).sum(axis=1))
score = d_neg / (d_pos + d_neg)
ranking = pd.DataFrame({"TOPSIS_score": score}).sort_values("TOPSIS_score", ascending=False)
ranking["rank"] = range(1, len(ranking) + 1)
ranking.to_csv(OUTPUT_ROOT / "topsis_ranking.csv", encoding="utf-8-sig")
ranking

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ranking["TOPSIS_score"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("TOPSIS comprehensive ranking")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "topsis_ranking.png", dpi=160)
plt.show()